In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from datetime import datetime
from torch.utils.data import IterableDataset, DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Getting data from the kaggle with the kagglehun method.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("crawford/emnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'emnist' dataset.
Path to dataset files: /kaggle/input/emnist


We are going to work with the bigger dataset to push our Neural Network to its limits, we are going to have around 7 Lakh images of letters and digits.

In [ ]:
image_path = f"{path}/emnist-bymerge-train.csv"

Streaming data directly from the kaggle and writing a custom Dataset which will itereate through and provide data in batches to the DataLoader and use those batches to train the model.

In [ ]:
class CustomStreamDataset(IterableDataset):
  def __init__(self, file_path, chunk_size=10000):
    self.file_path = file_path
    self.chunk_size = chunk_size

  def __len__(self):
    return len(self.x)

  def __iter__(self):
    for chunk in pd.read_csv(self.file_path, chunksize=self.chunk_size, header=None):
      y = chunk.iloc[:, 0].values
      x = chunk.iloc[:, 1:].values.astype('float32') / 255.0

      for i in range(len(x)):
        yield torch.tensor(x[i], dtype=torch.float32), torch.tensor(y[i], dtype=torch.long)


  def __getitem__(self, index):
    return self.x[index], self.y[index]

In [ ]:
dataset = CustomStreamDataset(image_path, 1000)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
first_batch = next(iter(loader))
first_batch[0].view(32, 1, 28, 28).shape
first_batch[0].shape

torch.Size([32, 784])

In [ ]:
class CNNCLassifier(nn.Module):
  def __init__(self):
    super().__init__()

    self.feature = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),

        nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),

        nn.MaxPool2d(2, 2),
        nn.Dropout2d(0.25),

        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),

        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),

        nn.MaxPool2d(2, 2),
        nn.Dropout2d(0.25)
    )


    self.classifer = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=3136, out_features=256, bias=False),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),

        nn.Linear(in_features=256, out_features=128, bias=False),
        nn.BatchNorm1d(128),
        nn.ReLU(inplace=True),

        nn.Dropout1d(0.5),
        nn.Linear(in_features=128, out_features=47),
    )

  def forward(self, x):
    return self.classifer(self.feature(x))

In [ ]:
model = CNNCLassifier().to(device=device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
from datetime import datetime

now = datetime.now()

for epoch in range(3):
  count = 0
  correct = 0
  epoch_loss = 0

  for x_batch, y_batch in loader:
    count += x_batch.size(0)
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    optimizer.zero_grad()
    pred = model(x_batch.view(x_batch.size(0), 1, 28, 28))

    loss = criterion(pred, y_batch)
    loss.backward()

    optimizer.step()

    epoch_loss += loss.item()

  print(f"Epoch {epoch + 1} | Loss {epoch_loss/count} | Time {datetime.now() - now } | Count {count}")





Epoch 1 | Loss 0.00846167483522692 | Time 0:02:29.843026 | Count 697932
Epoch 2 | Loss 0.008421547701004943 | Time 0:05:05.364091 | Count 697932
Epoch 3 | Loss 0.00836817816960411 | Time 0:07:36.954652 | Count 697932


In [ ]:
model.eval()
now = datetime.now()

correct = 0
total = 0

with torch.no_grad():
  test_loss = 0
  for x_batch, y_batch in loader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    pred = model(x_batch.view(x_batch.size(0), 1, 28, 28))
    something, predicted = torch.max(pred, 1)
    #print(f"Something {something}, predicted {predicted}")

    correct += (predicted == y_batch).sum().item()
    total += y_batch.size(0)
print(f"Accuracy {100 * correct / total:.2f}")

Accuracy 90.31


In [ ]:
image_path = f"{path}/emnist-bymerge-test.csv"
dataset = CustomStreamDataset(image_path, 1000)

test_loader = DataLoader(dataset, batch_size=32, shuffle=False)


In [ ]:
model.eval()

now = datetime.now()

correct = 0
total = 0

with torch.no_grad():
  test_loss = 0
  for x_batch, y_batch in test_loader:
    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    pred = model(x_batch.view(x_batch.size(0), 1, 28, 28))
    something, predicted = torch.max(pred, 1)

    correct += (predicted == y_batch).sum().item()
    total += y_batch.size(0)
  print(f"Accuracy {100 * correct / total:.2f}")

Accuracy 89.87
